# BOT experience data: read-only readiness checks
One rounds row = one completed non-warmup round. Events/signals join by round_id, never by player name. Set CSGO_EXPERIENCE_DB to override the local database path. This notebook never changes data or BOT decisions.
Historical rows have no build version or test-session marker. Descriptive results are not evidence that the new CT watch policy improved win rate; unmarked test sessions cannot be reliably excluded.

In [ ]:
import os, sqlite3, json
from pathlib import Path
from datetime import datetime, timezone, timedelta
db = Path(os.environ.get('CSGO_EXPERIENCE_DB', r'D:\CSGOServer\server\csgo\addons\sourcemod\data\sqlite\bot_tactical_experience.sq3')).resolve()
if not db.is_file(): raise FileNotFoundError(db)
connection = sqlite3.connect(db.as_uri() + '?mode=ro', uri=True)
connection.execute('PRAGMA query_only = ON')
connection.row_factory = sqlite3.Row
# One snapshot so concurrent recording cannot split the profile across states.
connection.execute('BEGIN')
def rows(sql): return [dict(row) for row in connection.execute(sql)]
print(json.dumps(rows("SELECT name,sql FROM sqlite_master WHERE type='table' AND name IN ('rounds','events','signals') ORDER BY name"), ensure_ascii=False, indent=2))

In [ ]:
queries = {
 'integrity': 'PRAGMA quick_check',
 'coverage': 'SELECT COUNT(*) AS rounds, MIN(started_at) AS first_epoch, MAX(started_at) AS last_epoch FROM rounds',
 'events_integrity': 'SELECT COUNT(*) AS events, SUM(CASE WHEN r.id IS NULL THEN 1 ELSE 0 END) AS orphan_events FROM events e LEFT JOIN rounds r ON r.id=e.round_id',
 'signals_integrity': 'SELECT COUNT(*) AS signals, SUM(CASE WHEN r.id IS NULL THEN 1 ELSE 0 END) AS orphan_signals, SUM(CASE WHEN elapsed_ms<0 THEN 1 ELSE 0 END) AS negative_elapsed FROM signals s LEFT JOIN rounds r ON r.id=s.round_id',
 'round_domains': "SELECT COUNT(*) AS invalid_rounds FROM rounds WHERE winner NOT IN (2,3) OR ct_buy NOT IN ('eco','force','full') OR t_buy NOT IN ('eco','force','full') OR round_index<0 OR ct_bots<0 OR t_bots<0",
 'signal_coverage': 'SELECT COUNT(*) AS rounds_with_signals FROM rounds r WHERE EXISTS (SELECT 1 FROM signals s WHERE s.round_id=r.id)',
 'maps': 'SELECT map,COUNT(*) AS rounds FROM rounds GROUP BY map ORDER BY rounds DESC,map',
 'daily': "SELECT date(started_at,'unixepoch','+8 hours') AS day,COUNT(*) AS rounds FROM rounds GROUP BY day ORDER BY day",
 'safe_preview': 'SELECT id,map,started_at,round_index,ct_buy,t_buy,winner FROM rounds ORDER BY id DESC LIMIT 3'
}
results = {name: rows(sql) for name,sql in queries.items()}
for field in ('first_epoch','last_epoch'):
 value=results['coverage'][0][field]
 results['coverage'][0][field.replace('_epoch','_shanghai')] = datetime.fromtimestamp(value, timezone(timedelta(hours=8))).isoformat() if value is not None else None
print(json.dumps(results, ensure_ascii=False, indent=2))
connection.rollback()
connection.close()

## Interpretation boundaries
- No orphan rows and valid domains support structural analysis, not event completeness. Missing signals may reflect an older recorder version.
- No version/session/test marker means before/after effects and training labels cannot be cleanly attributed. Do not delete or relabel historical rows to manufacture a clean experiment.
- Before enabling adaptive tactics, collect tagged, comparable, normal matches with map/economy/roster controls. Keep the current recorder read-only with respect to tactical decisions.